# AOI-driven access to remote COPC data

This notebook demonstrates the proof-of-concept architecture implemented in `PoC_aoi_copc.py`:

1. Load an AOI expressed in the same projected CRS as the AHN data.
2. Use DuckDB to find GeoParquet catalog rows whose tile bounding boxes intersect the AOI.
3. Pass each matched URL to the DuckDB PDAL community extension.
4. Apply the polygon as a `readers.copc` option so COPC hierarchy pruning happens before point data is downloaded.
5. Process one tile at a time to keep peak memory bounded.

In [ ]:
import json
import time
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

import duckdb
import pandas as pd
from shapely.geometry import shape

catalog_path = Path("ahn5.parquet")
aoi_path = Path("aoi.json")
requests = 4

# Use a coarse resolution for a quick demonstration. Set this to None to
# extract all source-resolution points inside the AOI.
resolution = 10.0
limit_tiles = 1

## 1. Load and validate the AOI

`aoi.json`, the catalog bounding boxes, and the COPC coordinates use RD New (EPSG:28992). GeoJSON does not reliably carry a projected CRS, so this CRS agreement is an explicit workflow precondition. Shapely supplies both WKT for PDAL and geometry validation.

In [ ]:
with aoi_path.open() as src:
    aoi_json = json.load(src)

aoi = shape(aoi_json["geometry"] if aoi_json.get("type") == "Feature" else aoi_json)
if aoi.is_empty or not aoi.is_valid:
    raise ValueError("AOI must be a non-empty, valid geometry")

aoi_wkt = aoi.wkt
aoi.bounds

## 2. Initialize DuckDB

The spatial extension handles tile-footprint intersection. The community PDAL extension exposes point-cloud readers as DuckDB table functions. DuckDB's `httpfs` extension is not required here because PDAL performs the COPC HTTP range requests itself.

In [ ]:
connection = duckdb.connect(database=":memory:")
connection.execute("INSTALL spatial; LOAD spatial;")
connection.execute("INSTALL pdal FROM community; LOAD pdal;")

## 3. Select intersecting COPC tiles

This query scans only the lightweight GeoParquet catalog. It constructs each tile footprint from the scalar bounding-box fields and intersects it with the AOI. No COPC point data is opened during this phase.

In [ ]:
catalog_query = """
    SELECT id, assets.data.href
    FROM read_parquet(?)
    WHERE ST_Intersects(
        ST_MakeEnvelope(bbox.xmin, bbox.ymin, bbox.xmax, bbox.ymax),
        ST_GeomFromText(?)
    )
    ORDER BY id
"""

matched_tiles = connection.execute(
    catalog_query, [str(catalog_path), aoi_wkt]
).fetchall()

print(f"Catalog matched {len(matched_tiles)} COPC tiles.")
pd.DataFrame(matched_tiles, columns=["tile_id", "url"])

## 4. Configure COPC reader-level filtering

AHN object names end in uppercase `.COPC.LAZ`, while PDAL 2.10 reader inference checks the lowercase `.copc.laz` suffix case-sensitively. Appending a lowercase URL fragment selects `readers.copc`; the fragment is not sent to the HTTP server.

`PDAL_Pipeline` is used instead of `PDAL_Read` because the current extension records the executed point count correctly for a spatially constrained pipeline. Its required downstream pipeline is a no-op expression. The polygon remains a reader option, which is essential for remote COPC hierarchy pruning.

In [ ]:
def force_copc_reader_url(url):
    parsed = urlsplit(url)
    if url.endswith(".copc.laz"):
        return url

    fragment = f"{parsed.fragment}/.copc.laz" if parsed.fragment else "/.copc.laz"
    return urlunsplit(parsed._replace(fragment=fragment))


filter_pipeline = json.dumps([
    {"type": "filters.expression", "expression": "X == X"}
])

if resolution is None:
    reader_options_sql = "MAP {'polygon': ?, 'requests': ?}"
else:
    reader_options_sql = "MAP {'polygon': ?, 'requests': ?, 'resolution': ?}"

## 5. Read and summarize one matched tile at a time

The PDAL extension currently materializes each selected `PointView` before returning DuckDB chunks. Running one statement per tile avoids retaining all matched point views simultaneously. The aggregate below demonstrates that points were extracted while returning only a compact result to Python.

In [ ]:
point_query = f"""
    WITH points AS (
        SELECT *
        FROM PDAL_Pipeline(?, ?, options => {reader_options_sql})
    )
    SELECT
        count(*) AS point_count,
        count_if(ST_Covers(ST_GeomFromText(?), ST_Point(X, Y))) AS points_inside_aoi,
        min(X) AS min_x, max(X) AS max_x,
        min(Y) AS min_y, max(Y) AS max_y,
        min(Z) AS min_z, max(Z) AS max_z
    FROM points
"""

summaries = []
tiles_to_process = matched_tiles[:limit_tiles] if limit_tiles else matched_tiles

for tile_id, url in tiles_to_process:
    parameters = [force_copc_reader_url(url), filter_pipeline, aoi_wkt, str(requests)]
    if resolution is not None:
        parameters.append(str(resolution))
    parameters.append(aoi_wkt)

    started = time.perf_counter()
    result = connection.execute(point_query, parameters).fetchone()
    elapsed = time.perf_counter() - started
    summaries.append((tile_id, *result, elapsed))

columns = [
    "tile_id", "point_count", "points_inside_aoi", "min_x", "max_x",
    "min_y", "max_y", "min_z", "max_z", "seconds"
]
summary_frame = pd.DataFrame(summaries, columns=columns)
if not (summary_frame["point_count"] == summary_frame["points_inside_aoi"]).all():
    raise RuntimeError("COPC reader returned points outside the AOI")

summary_frame

## 6. Confirm the run

At `resolution = 10.0` and `limit_tiles = 1`, this is a quick architecture check. Set `limit_tiles = None` to process every intersecting tile, and set `resolution = None` for all source-resolution points. The full AOI is large, so production processing should continue to aggregate or persist each tile before moving to the next one.

In [ ]:
print(f"Processed {summary_frame['point_count'].sum():,} AOI points.")